# RAVE Multilingual Training - Google Colab

Train RAVE (IRCAM) on multilingual voice data using the correct `acids-rave` CLI.

**GPU**: T4 (free tier) or better  
**Training time**: ~3-6 hours for 300k steps  
**Output**: exported `.ts` model ready for Max/MSP, SuperCollider, etc.

### Steps
1. Mount Drive & upload `training_data.wav`
2. Install `acids-rave`
3. Preprocess audio
4. Train
5. Export `.ts` model
6. Download results

## 1. Mount Google Drive & Copy Training Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

**Before running the next cell:** Upload `training_data.wav` to the root of your Google Drive at https://drive.google.com/

In [ ]:
import os
import shutil

DRIVE_WAV = '/content/drive/My Drive/training_data.wav'
DATA_DIR  = '/content/rave_data/audio'
PREP_DIR  = '/content/rave_data/preprocessed'
RUN_NAME  = 'multilingual_voice'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PREP_DIR, exist_ok=True)

dest = os.path.join(DATA_DIR, 'training_data.wav')
if os.path.exists(DRIVE_WAV):
    shutil.copy(DRIVE_WAV, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"Copied training_data.wav ({size_mb:.1f} MB) -> {dest}")
else:
    print(f"ERROR: {DRIVE_WAV} not found.")
    print("Upload training_data.wav to the root of your Google Drive and re-run.")

## 2. Install acids-rave

In [ ]:
!pip install -q acids-rave
!rave --help

## 3. Verify GPU

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")

## 4. Preprocess Audio

Slices the WAV into chunks and computes MFCC cache used during training.

In [ ]:
# Adjust --sr to match your training_data.wav sample rate (44100 or 48000)
!rave preprocess \
  --input_path  /content/rave_data/audio \
  --output_path /content/rave_data/preprocessed \
  --sr 44100

## 5. Train

Uses the `v2` config (recommended). Training saves checkpoints every N steps under `/content/rave_runs/<RUN_NAME>/`.  
**Typical target**: 300,000–500,000 steps for good quality. You can stop early and export any checkpoint.

In [ ]:
# --config options: v1, v2, v2_small (faster, less quality), discrete
# Remove --max_steps to train indefinitely (until you stop manually)
!rave train \
  --config v2 \
  --db_path /content/rave_data/preprocessed \
  --name    multilingual_voice \
  --max_steps 300000 \
  --val_every  10000

## 6. Find the Training Run Directory

In [ ]:
import glob
runs = glob.glob('/content/runs/multilingual_voice*')
if not runs:
    runs = glob.glob('/content/rave_runs/multilingual_voice*')
if not runs:
    # RAVE sometimes writes to cwd/runs
    runs = glob.glob('runs/multilingual_voice*')

print("Found runs:", runs)
RUN_DIR = runs[-1] if runs else None
print("Using:", RUN_DIR)

## 7. Export Model to TorchScript (`.ts`)

The exported `.ts` file is what you load in Max/MSP (nn~), SuperCollider (nn), or Python for inference.

In [ ]:
if RUN_DIR:
    !rave export --run "{RUN_DIR}" --cached False
    # Exported model appears next to the run directory as <name>.ts
    ts_files = glob.glob(f'{RUN_DIR}/../*.ts') + glob.glob(f'{RUN_DIR}/*.ts')
    print("Exported models:", ts_files)
else:
    print("No run directory found — check Step 6.")

## 8. Save to Google Drive & Download

In [ ]:
import shutil
from google.colab import files
import glob

drive_out = '/content/drive/My Drive/rave_output/'
os.makedirs(drive_out, exist_ok=True)

# Save exported model to Drive
ts_files = glob.glob(f'{RUN_DIR}/../*.ts') + glob.glob(f'{RUN_DIR}/*.ts')
for f in ts_files:
    shutil.copy(f, drive_out)
    print(f"Saved {f} -> Google Drive/rave_output/")

# Also save the run directory (checkpoints) to Drive so you can resume later
if RUN_DIR:
    dest_run = os.path.join(drive_out, os.path.basename(RUN_DIR))
    if not os.path.exists(dest_run):
        shutil.copytree(RUN_DIR, dest_run)
        print(f"Saved checkpoints -> Google Drive/rave_output/{os.path.basename(RUN_DIR)}/")

# Direct browser download of the .ts model
for f in ts_files:
    files.download(f)
print("Done.")